# درس ۱۸: تأمین امنیت عامل‌های هوش مصنوعی با رسیدهای رمزنگاری شده

## دفترچه تمرین تعاملی

این دفترچه از چهار تمرین عبور می‌کند:

۱. **امضای اولین رسید خود** برای فراخوانی یک ابزار عامل و تایید آن.
۲. **دستکاری در رسید** و مشاهده عدم موفقیت در تأیید.
۳. **ساخت زنجیره‌ای از سه رسید** و تأیید یکپارچگی زنجیره.
۴. **پیچیدن فراخوانی ابزار Microsoft Agent Framework** به گونه‌ای که هر عمل یک رسید صادر کند.

تمام توابع رمزنگاری از کتابخانه‌های به‌خوبی پشتیبانی شده وارد شده‌اند (`pynacl` برای Ed25519، `jcs` برای JSON کاننیکال RFC 8785، `hashlib` از کتابخانه استاندارد پایتون برای SHA-256). منطق رسید خود پایتون ساده است که می‌توانید آن را بخوانید و تغییر دهید.

سلول‌ها را به ترتیب اجرا کنید. هر بخش کوتاه و مستقل است.


## راه‌اندازی

دو وابستگی را نصب کنید. هر دو دارای مجوزهای پرمخاطب (Apache-2.0 / MIT) هستند.


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ابزارهای کمکی

این دو ابزار کمکی رمزگذاری base64url (بدون پر کردن) و هش SHA-256 اشیاء دلخواه را انجام می‌دهند. آنها کمک می‌کنند بقیه دفترچه به منطق رسید تمرکز داشته باشد.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## بخش ۱: اولین رسید خود را امضا کنید

تصور کنید نماینده ما برای **Contoso Travel** به تازگی پروازهایی از سیدنی به لس آنجلس را برای یک مشتری جستجو کرده است. ما می‌خواهیم این تماس با ابزار را به عنوان یک رسید امضا شده ثبت کنیم تا در آینده ممیزی‌کننده بتواند بدون اعتماد به ما آن را تأیید کند.

### مرحله ۱.۱: تولید کلید امضا

در محیط تولید، کلید امضای نماینده در یک ماژول امنیت سخت‌افزاری (HSM)، Azure Key Vault، یا مخزن محافظت شده مشابه ذخیره می‌شود. برای این درس، ما یک کلید جدید را در حافظه تولید می‌کنیم.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### مرحله ۱.۲: ساختن دادهٔ رسید

داده شامل همه چیزهایی است که می‌خواهیم رسید به آن گواهی دهد: چه کسی عمل کرده، چه ابزاری، با چه آرگومان‌هایی، چه نتیجه‌ای برگشت داده شده، بر اساس چه سیاستی، و چه زمانی. ما به جای درج مستقیم آرگومان‌ها و نتیجه، هش آنها را قرار می‌دهیم تا رسید محتوای حساس را فاش نکند.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### مرحله ۱.۳: امضا و مونتاژ رسید

سه مرحله:

۱. بارگذاری را با استفاده از JCS کانونیکال کنید تا دو پیاده‌سازی که رسید منطقی یکسانی تولید می‌کنند، بایت‌های کاملاً یکسانی ارائه دهند.
۲. بایت‌های کانونیکال شده را با SHA-256 هش کنید.
۳. هش را با کلید خصوصی Ed25519 امضا کنید.

سپس امضا به بارگذاری اصلی متصل می‌شود تا رسید نهایی تولید شود.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### مرحله ۱.۴: تایید رسید

تأیید روند را معکوس می‌کند. ما امضا را جدا می‌کنیم، هش قانونی را دوباره محاسبه می‌کنیم و امضا را با کلید عمومی موجود در رسید بررسی می‌کنیم.

ممیزی که این تایید را انجام می‌دهد، به جز خود رسید هیچ چیز دیگری از ما نیاز ندارد. نیازی به تماس با سرویس، پرس و جو در دایرکتوری کلیدها یا اعتماد نیست.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

شما باید `Receipt is valid: True` را ببینید. عامل نخستین رکورد حسابرسی امضاشده رمزنگاری‌شده خود را تولید کرده است.


## بخش ۲: دستکاری رسید

تمام هدف رسیدها این است که دستکاری آنها مشخص باشد. بیایید این موضوع را ثابت کنیم.

ما دقیقاً یک کاراکتر از رسید را تغییر می‌دهیم و شاهد شکست در اعتبارسنجی خواهیم بود.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### چه اتفاقی افتاد؟

وقتی `policy_id` را تغییر دادیم، بایت‌های استاندارد تغییر کردند. هش SHA-256 آن بایت‌ها تغییر کرد. امضایی که روی هش اصلی بود، دیگر با هش جدید مطابقت ندارد. اعتبارسنجی به درستی `False` برمی‌گرداند.

هیچ راهی برای تغییر هیچ فیلدی از رسید و در عین حال تأیید شدن آن وجود ندارد، مگر اینکه مهاجم کلید خصوصی را داشته باشد. تا زمانی که کلید خصوصی در یک خزانه کلید باشد و کلید عمومی منتشر شده باشد، مخدوش‌سازی غیرممکن است که مخفی بماند.

خودتان امتحان کنید: `tool_name` یا `agent_id` یا `timestamp` را در سلول بالا تغییر دهید و دوباره اجرا کنید. هر تغییر، یک رسید نامعتبر تولید می‌کند.


## بخش ۳: رسیدها را به هم زنجیر کنید

یک رسید واحد از یک اقدام محافظت می‌کند. اکثر عوامل چندین اقدام انجام می‌دهند. برای اینکه کل دنباله قابل تشخیص در برابر دستکاری باشد، هر رسید را با درج هش رسید قبلی در محتوای رسید جدید به رسید قبلی لینک می‌کنیم.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

اگر کسی رسیدی را حذف یا جابه‌جا کند، زنجیره دقیقاً در همان نقطه می‌شکند. تایید هر رسید بعدی ناموفق خواهد بود زیرا `previous_receipt_hash` دیگر با هش واقعی رسید قبلی مطابقت ندارد.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

اکنون زنجیره را با دستکاری رسید میانی بشکنید و دوباره بررسی کنید. رسید دستکاری شده در بررسی امضا رد می‌شود، و رسید بعدی در بررسی پیوند زنجیره‌ای شکست می‌خورد (زیرا `previous_receipt_hash` آن دیگر با هش رسید میانی دستکاری شده مطابقت ندارد).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

رسید ۰ هنوز تأیید می‌شود (تغییری نکرده است و پیش‌نیازی برای وابستگی ندارد). رسید ۱ بررسی امضای خود را رد می‌کند زیرا ما `tool_args_hash` را تغییر داده‌ایم. رسید ۲ بررسی پیوند زنجیره‌ای خود را رد می‌کند زیرا `previous_receipt_hash` آن بر اساس رسید اصلی (اکنون تغییر یافته) ۱ محاسبه شده است.

حتی اگر حمله‌کننده رسید تغییر یافته ۱ را دوباره امضا کند (که بدون کلید خصوصی نمی‌تواند این کار را انجام دهد)، ناهماهنگی پیوند زنجیره‌ای در رسید ۲ همچنان دستکاری را فاش می‌کند. برای مخفی کردن تغییر، حمله‌کننده باید همه رسیدها را از نقطه تغییر به بعد دوباره امضا کند، که مستلزم داشتن کلید خصوصی است.


## بخش ۴: فراخوانی ابزار عامل را با امضای رسید بسته‌بندی کنید

در یک استقرار واقعی، نمی‌خواهید هر نویسنده عامل به یاد داشته باشد که `make_receipt` را فرا بخواند. شما می‌خواهید امضای رسید به صورت خودکار برای هر فراخوانی ابزار انجام شود.

در اینجا ساده‌ترین الگو آورده شده است: یک کلاس پوشاننده که هر تابع ابزاری قابل فراخوانی را می‌گیرد و نسخه‌ای که رسید صادر می‌کند را برمی‌گرداند. این الگو با هر چارچوب عاملی سازگار است، از جمله چارچوب عامل مایکروسافت (`agent_framework.azure`).

اگر پروژه Azure AI Foundry را راه‌اندازی نکرده‌اید، الگوی محلی شبیه‌سازی شده در زیر همچنان این الگو را نشان می‌دهد.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### ادغام با چارچوب Microsoft Agent

قالب `ReceiptedTool` بالا مستقل از چارچوب است. برای استفاده از آن داخل یک عامل ساخته شده با چارچوب Microsoft Agent، تابع پیچیده شده را به‌عنوان یک ابزار ثبت کنید. یک طرح کلی (شما باید mock را با ثبت ابزار واقعی Azure AI Foundry جایگزین کنید):

```python
# شبه‌کد نشان‌دهنده شکل انتگرال‌گیری.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="شما یک نماینده سفر Contoso هستید ...",
#     tools=[receipted_lookup],   # ابزار بسته‌بندی شده، نه تابع خام
# )
# response = agent.run("پروازهای از سیدنی به لس آنجلس در ژوئن را پیدا کن.")
#
# # پس از اجرا، هر فراخوانی ابزاری که نماینده انجام داده، دارای رسید امضا شده است:
# audit_chain = receipted_lookup.receipts
```

چارچوب عامل نیازی به دانستن چیزی درباره رسیدها ندارد. امضای رسید به دور ابزار پیچیده شده است، نه اینکه در چارچوب تعبیه شود. این روش افزودن منبع به کد عامل موجود بدون بازنویسی عامل است.


## مرور و چالش گسترش

شما:

- یک جفت کلید Ed25519 تولید کرده‌اید.
- یک رسید برای فراخوانی ابزار عامل ساخته و امضا کرده‌اید.
- رسید را به‌صورت آفلاین فقط با کلید عمومی بررسی کرده‌اید.
- با رسید دستکاری کرده‌اید و مشاهده کرده‌اید که تاییدیه رد می‌شود.
- دنباله‌ای از سه رسید زنجیره‌ای هش ساخته‌اید.
- وسط زنجیره را دستکاری کرده‌اید و هر دو خطای امضا و شکست پیوند زنجیره‌ای را مشاهده کرده‌اید.
- یک تابع ابزار را با امضای خودکار رسید بسته‌بندی کرده‌اید.

**چالش گسترش.** طرح رسید را با یک فیلد `request_id` (یک UUID برای ردگیری توزیعی) گسترش دهید. `make_receipt` را به روز کنید تا آن را شامل شود، و تأیید کنید که رسیدها هنوز به صورت انتها به انتها بررسی می‌شوند. سپس پس از امضا، فیلد را تغییر دهید و تأیید رد شدن را تایید کنید. این شما را مجبور می‌کند که درک کنید هر بایت از کدگذاری قضایی چگونه به امضا کمک می‌کند.

**مرز مهم.** رسیدها سه چیز و فقط سه چیز را اثبات می‌کنند: نسبت دادن (این کلید این محتوا را امضا کرده است)، صحت (محتوا از زمان امضا تغییر نکرده است)، و ترتیب (این رسید بعد از آن رسید آمده است). آنها اثبات نمی‌کنند که عمل عامل درست بوده است، که سیاست نام‌برده‌شده در `policy_id` واقعاً ارزیابی شده است، یا عامل تمام قوانین را دنبال کرده است. رسیدها یک پایه هستند. حکمرانی سیستمی است که روی آن می‌سازید.

دوباره README درس را با این مرز در ذهن بخوانید. رایج‌ترین اشتباه تیم‌ها با رسیدها این است که فرض کنند «ما رسید داریم» یعنی «ما حکومت‌شده‌ایم.» این درست نیست. رسیدها رفتار عامل را قابل حسابرسی می‌کنند. آنها آن را درست نمی‌کنند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
